In [ ]:
# Resolve project inputs consistently when Jupyter opens this subdirectory.
import os

from paths import ROOT

os.chdir(ROOT)
# Set up plotting and the manuscript artifact directory.
import copy
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import wandb

figdir = Path("artifacts/dpx/figures/")
figdir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load the local W&B credential from .env so the notebook does not store secrets.
# python-dotenv reads the repo-root .env into os.environ, then wandb.Api uses that value.
import os

from dotenv import load_dotenv

load_dotenv(".env")
api = wandb.Api(api_key=os.environ["WANDB_API_KEY"])

entity = "chenghuzi"
project = "New_SPEL-Seq2"


gr_wo_RL_runs = api.runs(
    f"{entity}/{project}",
    # filters={
    #     "sim_config.tau": 100,
    #     "sim_config.predict_all": True,
    #     }
)

history = []
keys = ["global_steps", "correct_rate", "homeostatic_loss"]
# Collect data from all runs
for run in gr_wo_RL_runs:
    dfr = run.history(
        samples=50000,
        x_axis="global_steps",
        keys=keys,
        # keys=['name','global_steps', 'correct_rate', 'predictive_coding','homeostatic_loss']
    )
    # dfr = list(run.scan_history())
    # dfr = list(run.scan_history(keys=sorted(keys), page_size=500))
    # dfr = run.summary()
    dfr["predictive_coding"] = run.config["sim_config"]["predict_all"]
    dfr["name"] = run.name
    # dfr.set_index("global_steps", inplace=True)
    score = len(dfr)
    history.append(dfr)

In [ ]:
# Bin each original training history into 5000-trial windows.
dfs = copy.deepcopy(history)
global_steps_cut = 250000
window = 5000
df_ress = []
for df in dfs:
    bins = range(0, max(df["global_steps"]) + window, window)
    # Create bins for global_steps
    df["bins"] = pd.cut(df["global_steps"], bins=bins, right=False)

    # Group by bins and calculate the mean of 'another_col'
    df_res = (
        df.groupby("bins")
        .agg(
            {
                "correct_rate": "mean",
                "homeostatic_loss": "mean",
            }
        )
        .reset_index()
    )
    df_res["bin_start"] = df_res["bins"].apply(lambda x: x.left)
    df_res.set_index("bin_start", inplace=True)

    if df["predictive_coding"].iloc[0] == True:
        df_res["predictive_coding"] = "Predictive Coding RNN"
    else:
        df_res["predictive_coding"] = "Classical RNN"
    # df_res = df_res[df_res.index <=global_steps_cut]
    df_ress.append(df_res)
df_ress = pd.concat(df_ress)
# history[0]

In [ ]:
# Limit the training curves to the original manuscript display range.
# df_ress.index.name = 'Steps'
# df_ress.columns = ['bins', 'Correct Rate', 'Homeostatic Loss', 'predictive_coding']
# df_res[df_res.index <=global_steps_cut]
df_ress = df_ress[df_ress.index <= global_steps_cut]

In [ ]:
# Load the preserved reaction-time tables used for the behavioral comparison.
df_RT_pc_pre = pd.read_pickle("artifacts/dpx/predictive/rt_pre.pkl")
df_RT_pc_bal = pd.read_pickle("artifacts/dpx/predictive/rt_bal.pkl")
df_RT_cls_pre = pd.read_pickle("artifacts/dpx/classical/rt_pre.pkl")
df_RT_cls_bal = pd.read_pickle("artifacts/dpx/classical/rt_bal.pkl")

df_RT_pc = pd.concat(
    [
        df_RT_pc_pre
        #   , df_RT_pc_bal
    ]
)
df_RT_cls = pd.concat(
    [
        df_RT_cls_pre
        #    , df_RT_cls_bal
    ]
)

In [ ]:
# Assemble training, activity-loss, and reaction-time panels with architecture images.
df_res  # .plot(y='correct_rate')
color_map = {
    "Classical RNN": "#999999",  # Red
    "Predictive Coding RNN": "#d1352b",
}


fig, axes = plt.subplots(
    2,
    4,
    figsize=(9, 3.4),
    sharex="col",
    #  sharey='col'
    width_ratios=[2, 1, 1, 1],
)
ax = axes[0, 0]
ax.imshow(mpimg.imread("artifacts/dpx/assets/classical.jpg"))
ax.axis("off")

ax = axes[1, 0]
ax.imshow(mpimg.imread("artifacts/dpx/assets/predictive.jpg"))
ax.axis("off")


df_ress_classical = df_ress.loc[df_ress["predictive_coding"] == "Classical RNN"]
ax = axes[0, 1]
sns.lineplot(
    x="bin_start",
    y="correct_rate",
    hue="predictive_coding",
    #  style="event",
    data=df_ress_classical,
    ax=ax,
    legend=False,
    palette=color_map,
)
ax.set_xlabel("Steps")
ax.set_ylabel("Correct rate")

ax = axes[1, 1]
sns.lineplot(
    x="bin_start",
    y="correct_rate",
    hue="predictive_coding",
    #  style="event",
    data=df_ress,
    ax=ax,
    legend=False,
    palette=color_map,
)
ax.set_xlabel("Steps")
ax.set_ylabel("Correct rate")

ax = axes[0, 2]
sns.lineplot(
    x="bin_start",
    y="homeostatic_loss",
    hue="predictive_coding",
    data=df_ress_classical,
    ax=ax,
    legend=False,
    palette=color_map,
)
ax.set_xlabel("Steps")
ax.set_ylabel("Homeostatic loss")

ax = axes[1, 2]
sns.lineplot(
    x="bin_start",
    y="homeostatic_loss",
    hue="predictive_coding",
    data=df_ress,
    ax=ax,
    legend=False,
    palette=color_map,
)
ax.set_xlabel("Steps")
ax.set_ylabel("Homeostatic loss")

ax = axes[0, 3]
sns.boxplot(data=df_RT_cls, x="cue_probe", y="RT", ax=ax, color=color_map["Classical RNN"])
ax.set_xlabel("Trial type")
ax.set_ylabel("RT (ms)")


ax = axes[1, 3]
sns.boxplot(data=df_RT_pc, x="cue_probe", y="RT", ax=ax, color=color_map["Predictive Coding RNN"])
ax.set_xlabel("Trial type")
ax.set_ylabel("RT (ms)")

fig.tight_layout(rect=(0, 0, 1, 0.94), h_pad=3.0)
sns.despine()
fig.savefig(figdir / "fig2.jpg", dpi=300)

# Place compact bold panel labels above the upper-left corner of each subplot.
for ax, label in zip(axes.flat, "abcdefgh"):
    position = ax.get_position()
    fig.text(
        position.x0,
        position.y1 + 0.015,
        label,
        fontsize=14,
        fontweight="bold",
        color="black",
        ha="left",
        va="bottom",
    )

fig.savefig(figdir / "fig2.label.jpg", dpi=300)
fig.savefig(figdir / "fig2.label.png", dpi=300)

plt.show()